In [ ]:
# NOTEBOOK : 10_save_final_model.ipynb
# ROLE     : Compare Stacking vs best single model on test.csv and export the final production bundle.
# INPUT    : ../artifacts/metrics.json, ../artifacts/models/*.joblib, ../artifacts/features_list.json
# OUTPUT   : ../artifacts/models/match_model.joblib

In [1]:
from pathlib import Path
import json
import joblib

ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"

with open(ARTIFACTS_DIR / "metrics.json", "r", encoding="utf-8") as f:
    metrics_data = json.load(f)

with open(ARTIFACTS_DIR / "features_list.json", "r", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

scaler = joblib.load(MODELS_DIR / "match_scaler.joblib")

comparison = metrics_data["test_model_comparison"]
stacking_row = next(r for r in comparison if r["model"] == "stacking")
single_rows = [r for r in comparison if r["model"] != "stacking"]
best_single_row = min(single_rows, key=lambda r: r["log_loss"])

print(f"Best Single Model : {best_single_row['model']} (Test Log Loss = {best_single_row['log_loss']:.4f}, Acc = {best_single_row['accuracy']:.4f})")
print(f"Stacking Ensemble : stacking (Test Log Loss = {stacking_row['log_loss']:.4f}, Acc = {stacking_row['accuracy']:.4f})")

if stacking_row["log_loss"] < best_single_row["log_loss"]:
    final_choice = "stacking"
    reason = (
        f"Stacking achieved a lower test Log Loss ({stacking_row['log_loss']:.4f}) "
        f"than the best single model {best_single_row['model']} ({best_single_row['log_loss']:.4f})."
    )
else:
    final_choice = best_single_row["model"]
    reason = (
        f"Kept {best_single_row['model']} (Test Log Loss = {best_single_row['log_loss']:.4f}) over Stacking "
        f"(Test Log Loss = {stacking_row['log_loss']:.4f}). With a modest validation set size for Level-2 meta-learning, "
        f"the single model generalized better and avoids unnecessary ensemble complexity."
    )

print(f"\nFINAL DECISION: {final_choice}")
print(f"RATIONALE     : {reason}")

Best Single Model : logistic_regression (Test Log Loss = 1.0949, Acc = 0.4231)
Stacking Ensemble : stacking (Test Log Loss = 1.0793, Acc = 0.4385)

FINAL DECISION: stacking
RATIONALE     : Stacking achieved a lower test Log Loss (1.0793) than the best single model logistic_regression (1.0949).


In [2]:
SCALED_MODELS = {"logistic_regression"}

if final_choice == "stacking":
    model_obj = joblib.load(MODELS_DIR / "stacking_model.joblib")
    model_type = "stacking"
    uses_scaled = None  # handled per base model inside stacking_model dict
else:
    model_obj = joblib.load(MODELS_DIR / f"{final_choice}.joblib")
    model_type = "single"
    uses_scaled = final_choice in SCALED_MODELS

final_bundle = {
    "model_name": final_choice,
    "model_type": model_type,
    "uses_scaled": uses_scaled,
    "model": model_obj,
    "scaler": scaler,
    "features_list": FEATURE_COLS,
    "class_labels": {0: "A", 1: "D", 2: "H"},
    "selection_reason": reason,
    "test_metrics": stacking_row if final_choice == "stacking" else best_single_row,
}

final_path = MODELS_DIR / "match_model.joblib"
joblib.dump(final_bundle, final_path)
print(f"[SAVED] Production model bundle -> {final_path}")

[SAVED] Production model bundle -> ..\artifacts\models\match_model.joblib
